### News Topic Classification via Web Scraping and Text Mining

**Overview**

This project solves a news topic classification task with no provided training data. We first scrape labeled news articles from open news websites, using website sections as topic labels, and build our own training dataset.

Then we train a text classification model to predict one of nine target categories. Finally, we apply the trained model to the competition test set and generate a submission.csv file for Kaggle.


**Methodology**

Since the competition does not provide training data, we built our own labeled dataset by scraping news articles from publicly available news website sections corresponding to the target categories. Each article was assigned a label based on its rubric or section on the source website.

We collected article titles, full text, and topic labels, cleaned and normalized the text data, and stored the resulting dataset in CSV format. After that, we trained a supervised text classification model to predict one of the nine target classes.

The trained model was then applied to the competition test set, and the predictions were saved in the required CSV submission format.
Evaluation: The competition metric is accuracy, which measures the proportion of correctly predicted news topics.

**Evaluation**

The competition metric is accuracy, which measures the proportion of correctly predicted news topics.

**Variables**

- `train_df` — scraped training dataset
- `test_df` — competition test dataset
- `submission` — final prediction file
- `url` — article link
- `title` — news headline
- `content` — full news text
- `topic_name` — topic name as text label
- `topic` — numeric topic id
- `rubric` — website section used for scraping
- `links` — article links collected from a rubric page
- `article` — parsed news article object
- `pipeline` — full ML pipeline
- `x_train`, `x_valid` — train and validation features
- `y_train`, `y_valid` — train and validation labels
- `pred` — predicted topic ids


**Kaggle Competition Link**
https://www.kaggle.com/competitions/news-topics-competition-2025/overview

In [65]:
import re
import time
import random
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from scipy.sparse import hstack
from collections import Counter

import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

**1. Data Parsing**

In [3]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    )
}

In [4]:
LABEL_TO_ID = {
    "Общество/Россия": 0,
    "Экономика": 1,
    "Силовые структуры": 2,
    "Бывший СССР": 3,
    "Спорт": 4,
    "Забота о себе": 5,
    "Строительство": 6,
    "Туризм/Путешествия": 7,
    "Наука и техника": 8,
}

In [5]:
RUBRIC_TO_LABEL = {
    "russia": "Общество/Россия",
    "economics": "Экономика",
    "forces": "Силовые структуры",
    "formerussr": "Бывший СССР",
    "sport": "Спорт",
    "values": "Забота о себе",
    "house": "Строительство",
    "travel": "Туризм/Путешествия",
    "science": "Наука и техника",
}

In [6]:
@dataclass
class Article:
    url: str
    title: str
    content: str
    topic_name: str
    topic: int


def normalize_text(text: str) -> str:
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def fetch_soup(url: str, session: requests.Session, sleep_s: float = 0.2) -> Optional[BeautifulSoup]:
    try:
        response = session.get(url, headers=HEADERS, timeout=20)
        response.raise_for_status()
        time.sleep(sleep_s)
        return BeautifulSoup(response.text, "html.parser")
    except Exception:
        return None


def rubric_page_url(rubric: str, page: int) -> str:
    if page == 1:
        return f"https://lenta.ru/rubrics/{rubric}/"
    return f"https://lenta.ru/rubrics/{rubric}/page/{page}/"


def extract_article_links(soup: BeautifulSoup) -> List[str]:
    links = set()

    for anchor in soup.find_all("a", href=True):
        href = anchor["href"].strip()

        if href.startswith("//"):
            href = "https:" + href
        elif href.startswith("/"):
            href = "https://lenta.ru" + href

        if not href.startswith("https://lenta.ru/"):
            continue

        if re.search(r"/news/\d{4}/\d{2}/\d{2}/", href):
            links.add(href.split("#")[0].split("?")[0])

    return sorted(links)

In [7]:
def extract_title(soup: BeautifulSoup) -> str:
    selectors = [
        "h1",
        "span.topic-body__title",
        "div.topic-body__header h1",
        "h1.topic-body__title",
    ]
    for selector in selectors:
        node = soup.select_one(selector)
        if node:
            title = normalize_text(node.get_text(" ", strip=True))
            if title:
                return title
    return ""


def extract_content(soup: BeautifulSoup) -> str:
    selectors = [
        "div.topic-body__content-text p",
        "div.topic-body__content p",
        "div[itemprop='articleBody'] p",
        "article p",
    ]

    for selector in selectors:
        nodes = soup.select(selector)
        if nodes:
            paragraphs = [normalize_text(node.get_text(" ", strip=True)) for node in nodes]
            paragraphs = [p for p in paragraphs if p]
            if paragraphs:
                return " ".join(paragraphs)

    return ""


def parse_article(url: str, topic_name: str, session: requests.Session) -> Optional[Article]:
    soup = fetch_soup(url, session)
    if soup is None:
        return None

    title = extract_title(soup)
    content = extract_content(soup)

    if len(title) < 8 or len(content) < 200:
        return None

    return Article(
        url=url,
        title=title,
        content=content,
        topic_name=topic_name,
        topic=LABEL_TO_ID[topic_name],
    )

In [8]:
def scrape_lenta_dataset(
    pages_per_rubric: int = 60,
    max_articles_per_rubric: int = 1200,
) -> pd.DataFrame:
    session = requests.Session()
    rows: List[Dict] = []

    for rubric, topic_name in RUBRIC_TO_LABEL.items():
        seen_links = set()
        collected = 0
        print(f"\n[{rubric}] -> {topic_name}")

        for page in range(1, pages_per_rubric + 1):
            soup = fetch_soup(rubric_page_url(rubric, page), session)
            if soup is None:
                continue

            links = extract_article_links(soup)
            if not links:
                continue

            for link in tqdm(links, leave=False):
                if link in seen_links:
                    continue
                seen_links.add(link)

                article = parse_article(link, topic_name, session)
                if article is None:
                    continue

                rows.append(
                    {
                        "url": article.url,
                        "title": article.title,
                        "content": article.content,
                        "topic_name": article.topic_name,
                        "topic": article.topic,
                    }
                )
                collected += 1

                if collected >= max_articles_per_rubric:
                    break

            print(f"page={page:03d}, collected={collected}")
            if collected >= max_articles_per_rubric:
                break

    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    return df

In [9]:
train_df = scrape_lenta_dataset(
    pages_per_rubric=60,
    max_articles_per_rubric=1200,
)


[russia] -> Общество/Россия


page=001, collected=63

[economics] -> Экономика


page=001, collected=79

[forces] -> Силовые структуры


page=001, collected=81

[formerussr] -> Бывший СССР

[sport] -> Спорт


page=001, collected=79

[values] -> Забота о себе

[house] -> Строительство

[travel] -> Туризм/Путешествия


page=001, collected=78

[science] -> Наука и техника


page=001, collected=78


In [13]:
csv_name = "parsed_lenta_news.csv"
train_df.to_csv(csv_name, index=False, encoding="utf-8")

print(train_df.shape)
print(train_df.head())
print(f"Saved: {csv_name}")

(408, 5)
                                                 url  \
0  https://lenta.ru/news/2026/04/18/nad-rossiyski...   
1  https://lenta.ru/news/2026/04/18/novye-pdd-ust...   
2  https://lenta.ru/news/2026/04/18/oblomki-ukrai...   
3  https://lenta.ru/news/2026/04/18/pensioneram-n...   
4  https://lenta.ru/news/2026/04/18/posetitelnits...   

                                               title  \
0           Над российским городом прогремели взрывы   
1  Новые ПДД установят приоритет электровелосипед...   
2  Обломки украинского БПЛА рухнули возле российс...   
3      Пенсионерам назвали способы увеличить выплаты   
4  Посетительницу российского солярия оставили бе...   

                                             content       topic_name  topic  
0  Серия взрывов прогремела над Новокуйбышевском ...  Общество/Россия      0  
1  Изменения в правилах дорожного движения (ПДД),...  Общество/Россия      0  
2  Вооруженные силы Украины (ВСУ) продолжают попы...  Общество/Россия      0  
3

In [14]:
from google.colab import files
files.download(csv_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**2. Text Preprocessing**

In [16]:
df = pd.read_csv("parsed_lenta_news.csv")
print(df.shape)
df.head()

(408, 5)


,url,title,content,topic_name,topic
0,https://lenta.ru/news/2026/04/18/nad-rossiyski...,Над российским городом прогремели взрывы,Серия взрывов прогремела над Новокуйбышевском ...,Общество/Россия,0
1,https://lenta.ru/news/2026/04/18/novye-pdd-ust...,Новые ПДД установят приоритет электровелосипед...,"Изменения в правилах дорожного движения (ПДД),...",Общество/Россия,0
2,https://lenta.ru/news/2026/04/18/oblomki-ukrai...,Обломки украинского БПЛА рухнули возле российс...,Вооруженные силы Украины (ВСУ) продолжают попы...,Общество/Россия,0
3,https://lenta.ru/news/2026/04/18/pensioneram-n...,Пенсионерам назвали способы увеличить выплаты,"Многие россияне, отработавшие 35–40 лет, получ...",Общество/Россия,0
4,https://lenta.ru/news/2026/04/18/posetitelnits...,Посетительницу российского солярия оставили бе...,Неизвестный мужчина вломился в занятую кабинку...,Общество/Россия,0


In [17]:
def normalize_text(text):
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_text(text):
    text = normalize_text(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^0-9a-zа-яё\- ]+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [18]:
df["title"] = df["title"].fillna("").apply(clean_text)
df["content"] = df["content"].fillna("").apply(clean_text)
df["model_text"] = df["title"] + " [SEP] " + df["content"]

df[["title", "content", "model_text", "topic"]].head()

,title,content,model_text,topic
0,над российским городом прогремели взрывы,серия взрывов прогремела над новокуйбышевском ...,над российским городом прогремели взрывы [SEP]...,0
1,новые пдд установят приоритет электровелосипед...,изменения в правилах дорожного движения пдд ко...,новые пдд установят приоритет электровелосипед...,0
2,обломки украинского бпла рухнули возле российс...,вооруженные силы украины всу продолжают попытк...,обломки украинского бпла рухнули возле российс...,0
3,пенсионерам назвали способы увеличить выплаты,многие россияне отработавшие 35 40 лет получаю...,пенсионерам назвали способы увеличить выплаты ...,0
4,посетительницу российского солярия оставили бе...,неизвестный мужчина вломился в занятую кабинку...,посетительницу российского солярия оставили бе...,0


**3. Train / Validation Split**

In [19]:
X_train, X_valid, y_train, y_valid = train_test_split(
    df[["model_text"]],
    df["topic"],
    test_size=0.15,
    random_state=42,
    stratify=df["topic"]
)

print(X_train.shape, X_valid.shape)

(346, 1) (62, 1)


**4. Feature Extraction**

In [20]:
pipeline = Pipeline([
    (
        "features",
        ColumnTransformer(
            transformers=[
                (
                    "word_tfidf",
                    TfidfVectorizer(
                        analyzer="word",
                        ngram_range=(1, 2),
                        min_df=3,
                        max_df=0.98,
                        sublinear_tf=True,
                        max_features=250000,
                    ),
                    "model_text",
                ),
                (
                    "char_tfidf",
                    TfidfVectorizer(
                        analyzer="char_wb",
                        ngram_range=(3, 5),
                        min_df=3,
                        sublinear_tf=True,
                        max_features=250000,
                    ),
                    "model_text",
                ),
            ],
            remainder="drop",
            sparse_threshold=1.0,
        ),
    ),
    (
        "model",
        LinearSVC(C=1.2, random_state=42)
    )
])

**5. Model Training+Validation Evaluation**

In [21]:
pipeline.fit(X_train, y_train)
valid_pred = pipeline.predict(X_valid)

acc = accuracy_score(y_valid, valid_pred)
print("Validation accuracy:", acc)
print(classification_report(y_valid, valid_pred))

Validation accuracy: 0.8709677419354839
              precision    recall  f1-score   support

           0       0.88      0.70      0.78        10
           1       0.89      0.80      0.84        10
           2       0.77      0.91      0.83        11
           4       1.00      1.00      1.00        11
           7       1.00      0.90      0.95        10
           8       0.75      0.90      0.82        10

    accuracy                           0.87        62
   macro avg       0.88      0.87      0.87        62
weighted avg       0.88      0.87      0.87        62



**7. Predict Test Labels**

In [23]:
test_df = pd.read_csv("test_news.csv")
print(test_df.shape)
test_df.head()

(26275, 1)


,content
0,Фото: «Фонтанка.ру»ПоделитьсяЭкс-министру обор...
1,В начале февраля 2023 года в Пушкинском районе...
2,Фото: Andy Bao / Getty Images Анастасия Борисо...
3,"Если вы хотели, но так и не съездили на море л..."
4,Сергей Пиняев Фото: Алексей Филиппов / РИА Нов...


In [25]:
test_df["content"] = test_df["content"].fillna("").apply(clean_text)

test_inference = pd.DataFrame({
    "model_text": test_df["content"]
})

test_inference.head()

,model_text
0,фото фонтанка ру поделитьсяэкс-министру оборон...
1,в начале февраля 2023 года в пушкинском районе...
2,фото andy bao getty images анастасия борисова ...
3,если вы хотели но так и не съездили на море ле...
4,сергей пиняев фото алексей филиппов риа новост...


In [26]:
test_pred = pipeline.predict(test_inference)
test_pred[:10]

array([2, 1, 4, 7, 4, 0, 2, 1, 0, 0])

In [27]:
submission = pd.DataFrame({
    "topic": test_pred.astype(int),
    "index": np.arange(len(test_df))
})

submission.head()

,topic,index
0,2,0
1,1,1
2,4,2
3,7,3
4,4,4


In [28]:
submission.to_csv("submission.csv", index=False, encoding="utf-8")
print("submission.csv saved")

submission.csv saved


In [29]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Conclusion:** The accuracy score on a test data is weak (0.5081), trying to adjust the model.

**8. Amending the model**

In [30]:
def normalize_text(text):
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_text(text):
    text = normalize_text(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^0-9a-zа-яё\- ]+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [31]:
df["content"] = df["content"].fillna("").apply(clean_text)
df = df[df["content"].str.len() > 100].reset_index(drop=True)

X = df["content"]
y = df["topic"]

In [32]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

In [33]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    max_features=200000,
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),
    min_df=3,
    sublinear_tf=True,
    max_features=250000,
)

In [37]:
X_train_word = word_vectorizer.fit_transform(X_train)
X_valid_word = word_vectorizer.transform(X_valid)

X_train_char = char_vectorizer.fit_transform(X_train)
X_valid_char = char_vectorizer.transform(X_valid)

X_train_features = hstack([X_train_word, X_train_char])
X_valid_features = hstack([X_valid_word, X_valid_char])

In [38]:
svc_model = LinearSVC(C=1.5, random_state=42)
svc_model.fit(X_train_features, y_train)
svc_valid_pred = svc_model.predict(X_valid_features)

print("LinearSVC accuracy:", accuracy_score(y_valid, svc_valid_pred))

LinearSVC accuracy: 0.8709677419354839


In [41]:
lr_model = LogisticRegression(
    C=3.0,
    max_iter=3000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
lr_model.fit(X_train_features, y_train)
lr_valid_pred = lr_model.predict(X_valid_features)

print("LogisticRegression accuracy:", accuracy_score(y_valid, lr_valid_pred))


LogisticRegression accuracy: 0.8387096774193549


In [42]:
X_full_word = word_vectorizer.fit_transform(df["content"])
X_full_char = char_vectorizer.fit_transform(df["content"])
X_full_features = hstack([X_full_word, X_full_char])

svc_model = LinearSVC(C=1.5, random_state=42)
svc_model.fit(X_full_features, y)

lr_model = LogisticRegression(
    C=3.0,
    max_iter=3000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
lr_model.fit(X_full_features, y)

LogisticRegression(C=3.0, class_weight='balanced', max_iter=3000, n_jobs=-1,
                   random_state=42)

In [43]:
test_df = pd.read_csv("test_news.csv")
test_df["content"] = test_df["content"].fillna("").apply(clean_text)

In [46]:
X_test_word = word_vectorizer.transform(test_df["content"])
X_test_char = char_vectorizer.transform(test_df["content"])
X_test_features = hstack([X_test_word, X_test_char])

In [47]:
svc_test_pred = svc_model.predict(X_test_features)
lr_test_pred = lr_model.predict(X_test_features)

In [48]:
final_pred = []
for a, b in zip(svc_test_pred, lr_test_pred):
    if a == b:
        final_pred.append(a)
    else:
        final_pred.append(b)

final_pred = np.array(final_pred)

In [49]:
submission = pd.DataFrame({
    "topic": final_pred.astype(int),
    "index": np.arange(len(test_df), dtype=int)
})

submission.to_csv("submission.csv", index=False, encoding="utf-8")
print(submission.head())
print(submission.shape)

   topic  index
0      2      0
1      1      1
2      4      2
3      1      3
4      4      4
(26275, 2)


In [50]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**9. Amending the model #2**

In [51]:
df["title"] = df["title"].fillna("").apply(clean_text)
df["content"] = df["content"].fillna("").apply(clean_text)

df = df.drop_duplicates(subset=["content"]).reset_index(drop=True)
df = df[df["content"].str.len() > 300].reset_index(drop=True)
df = df[df["content"].str.len() < 12000].reset_index(drop=True)

print(df.shape)
print(df["topic"].value_counts())

(408, 6)
topic
2    71
1    69
4    69
7    68
8    68
0    63
Name: count, dtype: int64


In [52]:
# balance classes
min_class_size = df["topic"].value_counts().min()
balanced_parts = []

for cls in sorted(df["topic"].unique()):
    part = df[df["topic"] == cls].sample(min_class_size, random_state=42)
    balanced_parts.append(part)

df_balanced = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced.shape)
print(df_balanced["topic"].value_counts())

(378, 6)
topic
7    63
8    63
2    63
0    63
1    63
4    63
Name: count, dtype: int64


In [53]:
X = df_balanced["content"]
y = df_balanced["topic"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

In [54]:
X = df_balanced["content"]
y = df_balanced["topic"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

In [55]:
word_vec_1 = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 1),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    max_features=150000,
)

word_vec_2 = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    max_features=200000,
)

char_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),
    min_df=3,
    sublinear_tf=True,
    max_features=250000,
)

In [56]:
X_train_1 = word_vec_1.fit_transform(X_train)
X_valid_1 = word_vec_1.transform(X_valid)

X_train_2 = word_vec_2.fit_transform(X_train)
X_valid_2 = word_vec_2.transform(X_valid)

X_train_char = char_vec.fit_transform(X_train)
X_valid_char = char_vec.transform(X_valid)

In [57]:
F_train_a = hstack([X_train_1, X_train_char])
F_valid_a = hstack([X_valid_1, X_valid_char])

F_train_b = hstack([X_train_2, X_train_char])
F_valid_b = hstack([X_valid_2, X_valid_char])

In [58]:
model_a = LogisticRegression(
    C=4.0,
    max_iter=4000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model_a.fit(F_train_a, y_train)
pred_a = model_a.predict(F_valid_a)
print("Model A accuracy:", accuracy_score(y_valid, pred_a))

Model A accuracy: 0.8947368421052632


In [73]:
# prediction on test with LogisticRegression model_a

test_df = pd.read_csv("test_news.csv")
test_df["content"] = test_df["content"].fillna("").apply(clean_text)

X_test_1 = word_vec_1.transform(test_df["content"])
X_test_char = char_vec.transform(test_df["content"])
F_test_a = hstack([X_test_1, X_test_char])

test_pred_lr = model_a.predict(F_test_a)

submission = pd.DataFrame({
    "topic": test_pred_lr.astype(int),
    "index": np.arange(len(test_df), dtype=int)
})

submission.to_csv("submission_lr.csv", index=False, encoding="utf-8")
print(submission.head())
print(submission.shape)

   topic  index
0      2      0
1      1      1
2      4      2
3      1      3
4      4      4
(26275, 2)


In [74]:
from google.colab import files
files.download("submission_lr.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
model_b = LogisticRegression(
    C=2.0,
    max_iter=4000,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model_b.fit(F_train_b, y_train)
pred_b = model_b.predict(F_valid_b)
print("Model B accuracy:", accuracy_score(y_valid, pred_b))

Model B accuracy: 0.8771929824561403


In [61]:
from sklearn.linear_model import LogisticRegression, SGDClassifier

In [62]:
model_c = SGDClassifier(
    loss="log_loss",
    alpha=1e-5,
    max_iter=2000,
    random_state=42,
)
model_c.fit(F_train_b, y_train)
pred_c = model_c.predict(F_valid_b)
print("Model C accuracy:", accuracy_score(y_valid, pred_c))

Model C accuracy: 0.8771929824561403


In [66]:
# majority vote on validation
val_preds = np.vstack([pred_a, pred_b, pred_c]).T

final_valid_pred = []
for row in val_preds:
    final_valid_pred.append(Counter(row).most_common(1)[0][0])

final_valid_pred = np.array(final_valid_pred)
print("Ensemble accuracy:", accuracy_score(y_valid, final_valid_pred))

Ensemble accuracy: 0.8947368421052632


In [67]:
# retrain on full balanced train
X_full = df_balanced["content"]
y_full = df_balanced["topic"]

X_full_1 = word_vec_1.fit_transform(X_full)
X_full_2 = word_vec_2.fit_transform(X_full)
X_full_char = char_vec.fit_transform(X_full)

F_full_a = hstack([X_full_1, X_full_char])
F_full_b = hstack([X_full_2, X_full_char])

model_a.fit(F_full_a, y_full)
model_b.fit(F_full_b, y_full)
model_c.fit(F_full_b, y_full)

SGDClassifier(alpha=1e-05, loss='log_loss', max_iter=2000, random_state=42)

In [68]:
test_df = pd.read_csv("test_news.csv")
test_df["content"] = test_df["content"].fillna("").apply(clean_text)

X_test_1 = word_vec_1.transform(test_df["content"])
X_test_2 = word_vec_2.transform(test_df["content"])
X_test_char = char_vec.transform(test_df["content"])

F_test_a = hstack([X_test_1, X_test_char])
F_test_b = hstack([X_test_2, X_test_char])

In [69]:
test_pred_a = model_a.predict(F_test_a)
test_pred_b = model_b.predict(F_test_b)
test_pred_c = model_c.predict(F_test_b)

In [70]:
test_preds = np.vstack([test_pred_a, test_pred_b, test_pred_c]).T

final_test_pred = []
for row in test_preds:
    final_test_pred.append(Counter(row).most_common(1)[0][0])

final_test_pred = np.array(final_test_pred)

In [71]:
submission = pd.DataFrame({
    "topic": final_test_pred.astype(int),
    "index": np.arange(len(test_df), dtype=int)
})

submission.to_csv("submission3.csv", index=False, encoding="utf-8")
submission.head()

,topic,index
0,1,0
1,1,1
2,4,2
3,1,3
4,4,4


In [72]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Conclusion:** We built a full news topic classification pipeline, including web scraping, dataset creation, text preprocessing, model training, and submission generation. The baseline submission scored `0.50880`, the improved `TF-IDF` ensemble reached `0.53844`, and the **Logistic Regression** submission achieved the best public score of `0.54681`.

These results show that the approach is working, but also indicate that the main limitation comes from the quality and domain mismatch of the scraped training data rather than from the model itself.